In [1]:
"""
Pseudobulk DE — NO THRESHOLD (Comparison Run)
================================================
Same pipeline as pseudobulk_deseq2_all.py but WITHOUT any Cell2Location
abundance thresholding. All spots in each region (CT, IM, Stroma) are
included in the pseudobulk sum regardless of cell type composition.

Purpose: Compare results to the thresholded version to empirically
demonstrate the effect of abundance-based spot selection on DE results.

Output goes to:
  /Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/Differential_analysis/
    └── 00_No_Threshold_Comparison/
        ├── pseudobulk_counts.csv
        ├── pseudobulk_metadata.csv
        ├── DE_summary.csv
        ├── DE_<contrast>.csv
        ├── volcano_*.pdf
        └── run_log.txt

Run from your samples directory:
    cd /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples
    python pseudobulk_no_threshold.py
"""

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import warnings
from datetime import datetime
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import KDTree
from scipy import sparse

warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
VISIUM_DIR   = "/Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples"
META_PATH    = "/Users/adiallo/Desktop/Thesis/Data_Documents/data_all.csv"
RESULTS_ROOT = "/Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/Differential_analysis"
OUTPUT_DIR   = os.path.join(RESULTS_ROOT, "00_No_Threshold_Comparison")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# PARAMETERS
# ============================================================
MIN_SPOTS      = 20
MIN_GENES_EXPR = 10

# ============================================================
# REGION DEFINITION — from Define_regions.ipynb
# ============================================================
def define_regions_spatial(adata, pixel_scale_100um):
    coords = adata.obsm["spatial"]
    tumor_labels  = ["cancer_cancer"]
    stroma_labels = ["benign_stroma", "benign_fat", "inter", "stroma"]

    histo = adata.obs["histology"].astype(str)
    is_tumor  = histo.isin(tumor_labels).values
    is_stroma = histo.isin(stroma_labels).values

    region_labels = np.array(["Other"] * adata.n_obs, dtype=object)

    if is_tumor.sum() == 0:
        region_labels[is_stroma] = "Stroma"
        return region_labels
    if is_stroma.sum() == 0:
        region_labels[is_tumor] = "CT"
        return region_labels

    tumor_coords = coords[is_tumor]
    stroma_tree  = KDTree(coords[is_stroma])
    dists_to_stroma, _ = stroma_tree.query(tumor_coords, k=1)

    neighbor_threshold = 1.5 * pixel_scale_100um
    boundary_subset_mask = dists_to_stroma <= neighbor_threshold
    tumor_indices = np.where(is_tumor)[0]
    true_boundary_indices = tumor_indices[boundary_subset_mask]

    if len(true_boundary_indices) == 0:
        region_labels[is_tumor] = "CT"
        region_labels[is_stroma] = "Stroma"
        return region_labels

    boundary_coords = coords[true_boundary_indices]
    boundary_tree   = KDTree(boundary_coords)
    dists_to_boundary, _ = boundary_tree.query(coords)

    im_width_pixels = 5 * pixel_scale_100um
    is_proximal = dists_to_boundary <= im_width_pixels

    region_labels[is_proximal & (is_tumor | is_stroma)] = "IM"
    region_labels[is_tumor & (~is_proximal)] = "CT"
    region_labels[is_stroma & (~is_proximal)] = "Stroma"

    return region_labels


def classify_mets(row):
    any_mets = str(row["any_mets"]).strip().upper() in ["T", "TRUE", "1"]
    dist     = str(row["Distant_Mets"]).strip().upper() in ["T", "TRUE", "1"]
    if not any_mets: return "No_Mets"
    if dist: return "Distant_Mets"
    return "LN_Mets"


def make_volcano(ax, res_df, title_str):
    res = res_df.dropna(subset=["padj", "log2FoldChange"])
    if len(res) == 0:
        ax.set_title(f"{title_str}\n(no data)")
        return

    sig_up   = (res["padj"] < 0.05) & (res["log2FoldChange"] > 1)
    sig_down = (res["padj"] < 0.05) & (res["log2FoldChange"] < -1)
    ns       = ~(sig_up | sig_down)

    ax.scatter(res.loc[ns, "log2FoldChange"], -np.log10(res.loc[ns, "padj"]),
               c="gray", alpha=0.3, s=5)
    ax.scatter(res.loc[sig_up, "log2FoldChange"], -np.log10(res.loc[sig_up, "padj"]),
               c="#D62728", alpha=0.6, s=10, label=f"Up ({sig_up.sum()})")
    ax.scatter(res.loc[sig_down, "log2FoldChange"], -np.log10(res.loc[sig_down, "padj"]),
               c="#1F77B4", alpha=0.6, s=10, label=f"Down ({sig_down.sum()})")

    top_genes = res.head(10).index
    for gene in top_genes:
        if res.loc[gene, "padj"] < 0.05:
            ax.annotate(gene,
                        (res.loc[gene, "log2FoldChange"], -np.log10(res.loc[gene, "padj"])),
                        fontsize=5, alpha=0.8)

    ax.axhline(-np.log10(0.05), color="black", linestyle=":", alpha=0.3)
    ax.axvline(1, color="black", linestyle=":", alpha=0.3)
    ax.axvline(-1, color="black", linestyle=":", alpha=0.3)
    ax.set_xlabel("log2 Fold Change")
    ax.set_ylabel("-log10(padj)")
    ax.set_title(title_str)
    ax.legend(fontsize=8)


# ============================================================
# LOAD METADATA AND SAMPLES
# ============================================================
print("=" * 70)
print("Pseudobulk DE — NO THRESHOLD (All spots per region)")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

meta = pd.read_csv(META_PATH)
meta["deident"] = meta["deident"].astype(str)
meta["met_group"] = meta.apply(classify_mets, axis=1)

visium_files = sorted(
    f for f in os.listdir(VISIUM_DIR)
    if f.endswith(".h5ad") and f.replace(".h5ad", "").isdigit()
)
print(f"Found {len(visium_files)} sample files\n")

# ============================================================
# LOAD ALL SAMPLES
# ============================================================
print("Loading all samples...")
print("-" * 40)

sample_cache = {}

for i, f in enumerate(visium_files):
    sid = f.replace(".h5ad", "")
    print(f"  [{i+1}/{len(visium_files)}] {sid}...", end=" ")

    try:
        adata = sc.read_h5ad(os.path.join(VISIUM_DIR, f))

        if "histology" not in adata.obs.columns:
            print("SKIPPED (no histology)")
            continue

        gene_names = adata.var_names.tolist()
        X = adata.X
        if sparse.issparse(X):
            X = X.toarray()

        coords = adata.obsm["spatial"]
        nbrs = NearestNeighbors(n_neighbors=2).fit(coords)
        distances, _ = nbrs.kneighbors(coords)
        pixel_scale = np.median(distances[:, 1])
        region_labels = define_regions_spatial(adata, pixel_scale)

        row = meta.loc[meta["deident"] == sid]
        if row.shape[0] != 1:
            print("SKIPPED (metadata mismatch)")
            continue
        clin = row.iloc[0]

        lib_sizes = np.asarray(X.sum(axis=1)).flatten()

        sample_cache[sid] = {
            "X": X,
            "gene_names": gene_names,
            "region_labels": region_labels,
            "clin": clin,
            "lib_sizes": lib_sizes,
        }

        n_ct = (region_labels == "CT").sum()
        n_im = (region_labels == "IM").sum()
        n_str = (region_labels == "Stroma").sum()
        print(f"OK (CT={n_ct}, IM={n_im}, Stroma={n_str})")

    except Exception as e:
        print(f"FAILED: {e}")

print(f"\nCached {len(sample_cache)} samples\n")

# Shared genes
all_gene_lists = [s["gene_names"] for s in sample_cache.values()]
shared_genes = list(set(all_gene_lists[0]))
for gl in all_gene_lists[1:]:
    shared_genes = [g for g in shared_genes if g in gl]
shared_genes = sorted(shared_genes)
print(f"Shared genes: {len(shared_genes)}")

for sid, data in sample_cache.items():
    gene_idx = [data["gene_names"].index(g) for g in shared_genes]
    data["gene_idx"] = gene_idx

# ============================================================
# PSEUDOBULK AGGREGATION — NO THRESHOLD
# All spots in each region are summed, no cell type filtering
# ============================================================
print("\nAggregating pseudobulk — NO THRESHOLD")
print("-" * 40)

pseudobulk_counts = {}
pseudobulk_meta = []

for sid, data in sample_cache.items():
    X_shared = data["X"][:, data["gene_idx"]]
    region_labels = data["region_labels"]
    clin = data["clin"]

    for region in ["CT", "IM", "Stroma"]:
        mask = region_labels == region  # NO THRESHOLD — all spots in region
        n_spots = mask.sum()

        if n_spots < MIN_SPOTS:
            continue

        counts_sum = X_shared[mask, :].sum(axis=0)
        counts_sum = np.asarray(counts_sum).flatten()

        sample_key = f"{sid}_{region}"
        pseudobulk_counts[sample_key] = counts_sum

        mean_lib = np.log1p(np.mean(data["lib_sizes"][mask]))

        pseudobulk_meta.append({
            "sample_key": sample_key,
            "patient_id": sid,
            "region": region,
            "mets_status": classify_mets(clin),
            "MLH1": int(clin["MLH1"]),
            "age": int(clin["age"]),
            "sex": clin["sex"],
            "n_spots": n_spots,
            "QC": mean_lib,
        })

meta_df = pd.DataFrame(pseudobulk_meta).set_index("sample_key")
count_df = pd.DataFrame(pseudobulk_counts, index=shared_genes).T
count_df = count_df.loc[meta_df.index].astype(int)

# Filter genes
genes_detected = (count_df > 0).sum(axis=0)
keep_genes = genes_detected[genes_detected >= MIN_GENES_EXPR].index
count_df = count_df[keep_genes]

print(f"Pseudobulk samples: {count_df.shape[0]}")
print(f"Genes: {count_df.shape[1]}")
print(f"\nRegion × Mets:")
print(pd.crosstab(meta_df["region"], meta_df["mets_status"]))

count_df.to_csv(os.path.join(OUTPUT_DIR, "pseudobulk_counts_no_threshold.csv"))
meta_df.to_csv(os.path.join(OUTPUT_DIR, "pseudobulk_metadata_no_threshold.csv"))

# ============================================================
# RUN pyDESeq2
# ============================================================
print("\nRunning pyDESeq2...")
print("-" * 40)

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

meta_df["group"] = meta_df["region"] + "_" + meta_df["mets_status"]
meta_df["group"] = pd.Categorical(meta_df["group"])
meta_df["sex"] = pd.Categorical(meta_df["sex"])
meta_df["MLH1"] = meta_df["MLH1"].astype(int)

try:
    dds = DeseqDataSet(
        counts=count_df,
        metadata=meta_df,
        design="~ QC + MLH1 + age + sex + group"
    )
    dds.deseq2()
except Exception as e:
    print(f"DESeq2 FAILED: {e}")
    exit()

# Contrasts
contrasts = {
    "IM_vs_CT__No_Mets": ["group", "IM_No_Mets", "CT_No_Mets"],
    "IM_vs_CT__LN_Mets": ["group", "IM_LN_Mets", "CT_LN_Mets"],
    "IM_vs_CT__Distant_Mets": ["group", "IM_Distant_Mets", "CT_Distant_Mets"],
    "Distant_vs_No__CT": ["group", "CT_Distant_Mets", "CT_No_Mets"],
    "Distant_vs_No__IM": ["group", "IM_Distant_Mets", "IM_No_Mets"],
    "Distant_vs_No__Stroma": ["group", "Stroma_Distant_Mets", "Stroma_No_Mets"],
    "LN_vs_No__CT": ["group", "CT_LN_Mets", "CT_No_Mets"],
    "LN_vs_No__IM": ["group", "IM_LN_Mets", "IM_No_Mets"],
    "LN_vs_No__Stroma": ["group", "Stroma_LN_Mets", "Stroma_No_Mets"],
    "Stroma_vs_CT__No_Mets": ["group", "Stroma_No_Mets", "CT_No_Mets"],
    "Stroma_vs_IM__No_Mets": ["group", "Stroma_No_Mets", "IM_No_Mets"],
}

results_all = {}
summary_rows = []

for cname, contrast in contrasts.items():
    if contrast[1] not in meta_df["group"].values or contrast[2] not in meta_df["group"].values:
        print(f"  SKIPPED {cname}: group missing")
        continue

    try:
        stat_res = DeseqStats(dds, contrast=contrast, alpha=0.05)
        stat_res.summary()
        res_df = stat_res.results_df.sort_values("padj")
        results_all[cname] = res_df

        n_sig  = (res_df["padj"] < 0.05).sum()
        n_up   = ((res_df["padj"] < 0.05) & (res_df["log2FoldChange"] > 0)).sum()
        n_down = ((res_df["padj"] < 0.05) & (res_df["log2FoldChange"] < 0)).sum()

        summary_rows.append({
            "contrast": cname,
            "n_sig_005": n_sig,
            "n_up": n_up,
            "n_down": n_down,
            "top_gene": res_df.index[0] if n_sig > 0 else "none",
            "top_padj": res_df["padj"].iloc[0],
            "top_lfc": res_df["log2FoldChange"].iloc[0],
        })

        print(f"  {cname}: {n_sig} DE genes (↑{n_up} ↓{n_down})")
        res_df.to_csv(os.path.join(OUTPUT_DIR, f"DE_{cname}.csv"))

    except Exception as e:
        print(f"  FAILED {cname}: {e}")

# ============================================================
# VOLCANO PLOTS
# ============================================================
volcano_groups = {
    "IM_vs_CT_by_mets": {
        "title": "NO THRESHOLD — IM vs CT by Mets",
        "contrasts": ["IM_vs_CT__No_Mets", "IM_vs_CT__LN_Mets", "IM_vs_CT__Distant_Mets"],
    },
    "Distant_vs_No_by_region": {
        "title": "NO THRESHOLD — Distant vs No Mets by Region",
        "contrasts": ["Distant_vs_No__CT", "Distant_vs_No__IM", "Distant_vs_No__Stroma"],
    },
    "LN_vs_No_by_region": {
        "title": "NO THRESHOLD — LN vs No Mets by Region",
        "contrasts": ["LN_vs_No__CT", "LN_vs_No__IM", "LN_vs_No__Stroma"],
    },
}

for group_key, group_info in volcano_groups.items():
    contrast_list = [c for c in group_info["contrasts"] if c in results_all]
    if len(contrast_list) == 0:
        continue

    ncols = len(contrast_list)
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))
    fig.suptitle(group_info["title"], fontsize=13, fontweight="bold")

    if ncols == 1:
        axes = [axes]

    for ax, cn in zip(axes, contrast_list):
        display_name = cn.replace("__", " | ").replace("_", " ")
        make_volcano(ax, results_all[cn], display_name)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"volcano_{group_key}.png"), dpi=150, bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_DIR, f"volcano_{group_key}.pdf"), bbox_inches="tight")
    plt.close()

# ============================================================
# SUMMARY
# ============================================================
if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(OUTPUT_DIR, "DE_summary_no_threshold.csv"), index=False)
    print("\nDE Summary (NO THRESHOLD):")
    print(summary_df.to_string(index=False))

# Save log
with open(os.path.join(OUTPUT_DIR, "run_log.txt"), "w") as f:
    f.write(f"Run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Method: NO THRESHOLD — all spots in each region included\n")
    f.write(f"Min spots: {MIN_SPOTS}\n")
    f.write(f"Design: ~ QC + MLH1 + age + sex + group\n")
    f.write(f"  where group = region × mets_status\n\n")
    f.write(f"NOTE: This is a comparison run. Without thresholding,\n")
    f.write(f"the pseudobulk for each region captures the WHOLE tissue\n")
    f.write(f"transcriptome, not cell-type-specific signal. Any DE genes\n")
    f.write(f"reflect regional differences in overall tissue composition\n")
    f.write(f"rather than cell-type-specific transcriptomic changes.\n")

print(f"\nResults saved to: {OUTPUT_DIR}/")
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n" + "=" * 70)
print("NEXT: Compare these results to the thresholded cell-type-specific")
print("results to demonstrate why abundance thresholding matters.")
print("=" * 70)

Pseudobulk DE — NO THRESHOLD (All spots per region)
Started: 2026-03-07 23:04:18
Found 41 sample files

Loading all samples...
----------------------------------------
  [1/41] 0... OK (CT=2901, IM=1308, Stroma=2080)
  [2/41] 10... OK (CT=3400, IM=585, Stroma=1787)
  [3/41] 100... OK (CT=6992, IM=168, Stroma=0)
  [4/41] 102... OK (CT=687, IM=0, Stroma=3681)
  [5/41] 106... OK (CT=4256, IM=1103, Stroma=94)
  [6/41] 107... OK (CT=3206, IM=366, Stroma=2908)
  [7/41] 11... OK (CT=4787, IM=903, Stroma=68)
  [8/41] 111... OK (CT=4913, IM=1073, Stroma=838)
  [9/41] 116... OK (CT=6106, IM=965, Stroma=83)
  [10/41] 117... OK (CT=4562, IM=126, Stroma=18)
  [11/41] 119... OK (CT=1322, IM=1003, Stroma=56)
  [12/41] 122... OK (CT=3826, IM=1002, Stroma=0)
  [13/41] 127... OK (CT=6426, IM=378, Stroma=90)
  [14/41] 128... OK (CT=5012, IM=0, Stroma=0)
  [15/41] 13... OK (CT=2949, IM=784, Stroma=226)
  [16/41] 14... OK (CT=2728, IM=2797, Stroma=168)
  [17/41] 18... OK (CT=1663, IM=1654, Stroma=78)
  [18

Fitting size factors...
... done in 0.03 seconds.

/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:378: RuntimeWarning: overflow encountered in exp
  mu_ = np.maximum(size_factors * np.exp(X @ beta), min_mu)
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:232: RuntimeWarning: invalid value encountered in multiply
  - counts * np.log(mu)
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:230: RuntimeWarning: invalid value encountered in subtract
  -logbinom
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:382: RuntimeWarning: overflow encountered in exp
  mu_ = np.maximum(size_factors * np.exp(X @ beta), min_mu)
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:385: RuntimeWarning: invalid value encountered in divide
  + ((1 / disp + counts) * mu_ / (1 / disp + mu_)) @ X
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:378: RuntimeWarning: 

Log2 fold change & Wald test p-value: group IM_No_Mets vs CT_No_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1CF     222.899906       -0.687675  0.401285 -1.713682  0.086587  0.369635
A2M     7015.936811        1.092185  0.284481  3.839216  0.000123  0.007629
A4GALT   471.427876        0.752941  0.265970  2.830922  0.004641  0.068305
AAAS     831.106369        0.059944  0.119982  0.499607  0.617351  0.828397
AACS     618.358784       -0.305061  0.142946 -2.134093  0.032835  0.228035
...             ...             ...       ...       ...       ...       ...
ZWINT    692.961732       -0.291925  0.256006 -1.140308  0.254158  0.572674
ZXDC     537.452932       -0.262649  0.089029 -2.950152  0.003176  0.053438
ZYG11B   608.523205        0.037488  0.103545  0.362045  0.717319  0.881120
ZYX     4263.791405        0.292854  0.182592  1.603872  0.108742  0.406266
ZZEF1    761.370777        0.110857  0.128507  0.862656  0.388326  0.683510

[13598 rows x 6 co

... done in 0.39 seconds.

Running Wald tests...


Log2 fold change & Wald test p-value: group IM_LN_Mets vs CT_LN_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1CF     222.899906       -0.734941  0.481391 -1.526703  0.126835  0.339838
A2M     7015.936811        0.832776  0.340760  2.443880  0.014530  0.122613
A4GALT   471.427876        0.876030  0.317586  2.758399  0.005809  0.083589
AAAS     831.106369       -0.077315  0.143206 -0.539890  0.589273  0.764383
AACS     618.358784       -0.549077  0.171192 -3.207376  0.001340  0.046661
...             ...             ...       ...       ...       ...       ...
ZWINT    692.961732       -0.611014  0.306352 -1.994486  0.046099  0.209281
ZXDC     537.452932       -0.092879  0.106318 -0.873596  0.382338  0.608075
ZYG11B   608.523205       -0.045093  0.123381 -0.365475  0.714757  0.846338
ZYX     4263.791405       -0.017149  0.218619 -0.078445  0.937474  0.972073
ZZEF1    761.370777        0.180479  0.153489  1.175840  0.239659  0.470674

[13598 rows x 6 co

...

Log2 fold change & Wald test p-value: group Stroma_LN_Mets vs Stroma_No_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1CF     222.899906        0.419769  0.466360  0.900097  0.368069  0.999901
A2M     7015.936811        0.224287  0.314255  0.713711  0.475406  0.999901
A4GALT   471.427876        0.076821  0.296472  0.259118  0.795545  0.999901
AAAS     831.106369        0.089042  0.139282  0.639292  0.522633  0.999901
AACS     618.358784       -0.035514  0.169332 -0.209727  0.833881  0.999901
...             ...             ...       ...       ...       ...       ...
ZWINT    692.961732       -0.367343  0.292663 -1.255174  0.209416  0.999901
ZXDC     537.452932        0.123456  0.111091  1.111297  0.266440  0.999901
ZYG11B   608.523205        0.037906  0.122477  0.309498  0.756943  0.999901
ZYX     4263.791405       -0.054013  0.202418 -0.266841  0.789592  0.999901
ZZEF1    761.370777       -0.088900  0.148275 -0.599560  0.548800  0.999901

[13598 row

... done in 0.39 seconds.

Running Wald tests...


Log2 fold change & Wald test p-value: group Stroma_No_Mets vs CT_No_Mets
           baseMean  log2FoldChange     lfcSE      stat        pvalue  \
A1CF     222.899906       -1.539044  0.469704 -3.276624  1.050563e-03   
A2M     7015.936811        1.923697  0.322429  5.966264  2.427473e-09   
A4GALT   471.427876        1.358454  0.303839  4.470969  7.786599e-06   
AAAS     831.106369        0.067920  0.140280  0.484174  6.282622e-01   
AACS     618.358784       -0.789362  0.168834 -4.675382  2.934067e-06   
...             ...             ...       ...       ...           ...   
ZWINT    692.961732       -1.077871  0.295414 -3.648682  2.635889e-04   
ZXDC     537.452932       -0.500782  0.108648 -4.609217  4.041883e-06   
ZYG11B   608.523205        0.254665  0.122652  2.076312  3.786515e-02   
ZYX     4263.791405        0.700294  0.207422  3.376175  7.350125e-04   
ZZEF1    761.370777        0.287479  0.149632  1.921240  5.470145e-02   

                padj  
A1CF    4.370006e-03  
A2M 

... done in 0.35 seconds.



Log2 fold change & Wald test p-value: group Stroma_No_Mets vs IM_No_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1CF     222.899906       -0.851369  0.427089 -1.993422  0.046215  0.154290
A2M     7015.936811        0.831512  0.289594  2.871304  0.004088  0.034303
A4GALT   471.427876        0.605513  0.273525  2.213741  0.026847  0.110417
AAAS     831.106369        0.007976  0.127531  0.062543  0.950131  0.974392
AACS     618.358784       -0.484301  0.153803 -3.148839  0.001639  0.020572
...             ...             ...       ...       ...       ...       ...
ZWINT    692.961732       -0.785946  0.266804 -2.945782  0.003221  0.029749
ZXDC     537.452932       -0.238134  0.100341 -2.373250  0.017632  0.084938
ZYG11B   608.523205        0.217177  0.112183  1.935909  0.052879  0.167998
ZYX     4263.791405        0.407440  0.186484  2.184857  0.028899  0.115338
ZZEF1    761.370777        0.176622  0.135864  1.299987  0.193606  0.384063

[13598 rows x 